In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from scipy.stats import norm

In [11]:
DATA_ROOT = Path("data/Murphy2021/raw")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("~/Documents/academic/data/Murphy2021/raw").expanduser()

INCLUDE_TRAINING = False
PARTICIPANTS = None  # Set to a list like ["DCB"] to run a subset.

mean_neg, mean_pos = -17, 17
std = 29
P0 = np.array([[0.5], [0.5]])
h = 0.08

In [3]:
def load_mat(path):
    """Load a MATLAB .mat file and drop scipy's private metadata keys."""
    mat = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
    return {key: value for key, value in mat.items() if not key.startswith("__")}


def matlab_value_to_python(value):
    """Convert MATLAB scalar/array/struct values into notebook-friendly Python objects."""
    if hasattr(value, "_fieldnames"):
        return {field: matlab_value_to_python(getattr(value, field)) for field in value._fieldnames}

    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            return matlab_value_to_python(value.item())
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    return value


def flatten_dict(d, prefix=""):
    """Flatten nested metadata dictionaries into dot-separated columns."""
    rows = {}
    for key, value in d.items():
        name = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):
            rows.update(flatten_dict(value, name))
        else:
            rows[name] = value
    return rows


def parse_subject_session_block(path):
    """Read participant/session/block from file names like DCB_1_1.mat."""
    parts = path.stem.split("_")
    block = int(parts[-1])
    session_number = int(parts[-2])
    participant_id = "_".join(parts[:-2])
    return participant_id, session_number, block


def discover_participants(data_root=DATA_ROOT, participants=PARTICIPANTS):
    if participants is not None:
        return list(participants)
    return sorted(path.name for path in data_root.iterdir() if path.is_dir())


In [4]:
def load_subject_behavior(subject_folder, data_root=DATA_ROOT, include_training=INCLUDE_TRAINING):
    subject_dir = data_root / subject_folder
    session_dirs = sorted(
        path
        for path in subject_dir.iterdir()
        if path.is_dir() and (include_training or path.name != "Training")
    )

    behav_columns = [
        "correct_response_dist",
        "response",
        "accuracy",
        "rt_from_go_cue",
        "trial_onset_ptb",
        "fixation_break_count",
    ]

    trial_frames = []
    sample_frames = []
    block_metadata_rows = []

    for session_dir in session_dirs:
        behaviour_dir = session_dir / "Behaviour"
        sample_seqs_dir = session_dir / "Sample_seqs"

        for behav_path in sorted(behaviour_dir.glob("*.mat")):
            sample_path = sample_seqs_dir / behav_path.name
            if not sample_path.exists():
                raise FileNotFoundError(f"Missing matching Sample_seqs file for {behav_path}")

            participant_id, session_number, block = parse_subject_session_block(behav_path)
            block_id = f"{participant_id}_S{session_number}_B{block}"

            behav_mat = load_mat(behav_path)
            sample_mat = load_mat(sample_path)

            behav = np.asarray(behav_mat["Behav"])
            t_refresh = np.asarray(behav_mat["tRefresh"])
            stim_in = np.asarray(sample_mat["stimIn"])
            distseqs = np.asarray(sample_mat["distseqs"])
            pswitch = np.asarray(sample_mat["pswitch"])

            if behav.ndim != 2 or behav.shape[1] != len(behav_columns):
                raise ValueError(f"Unexpected Behav shape in {behav_path}: {behav.shape}")

            n_planned_trials, n_samples = stim_in.shape
            n_trials = behav.shape[0]
            if n_trials > n_planned_trials:
                raise ValueError(
                    f"More behavioral trials than stimulus rows for {block_id}: "
                    f"Behav has {n_trials}, stimIn has {n_planned_trials}"
                )

            stim_in_completed = stim_in[:n_trials]
            distseqs_completed = distseqs[:n_trials]
            pswitch_completed = pswitch[:n_trials]
            t_refresh_completed = t_refresh[:n_trials]

            trial_df = pd.DataFrame(behav, columns=behav_columns)
            trial_df.insert(0, "trial", np.arange(1, n_trials + 1))
            trial_df.insert(0, "block", block)
            trial_df.insert(0, "session", session_number)
            trial_df.insert(0, "session_label", session_dir.name)
            trial_df.insert(0, "participant_id", participant_id)
            trial_df.insert(0, "subject_folder", subject_dir.name)
            trial_df.insert(0, "block_id", block_id)

            trial_df["n_presented_samples"] = np.sum(~np.isnan(stim_in_completed), axis=1)
            trial_df["final_sample_location"] = stim_in_completed[
                np.arange(n_trials), trial_df["n_presented_samples"].to_numpy() - 1
            ]
            trial_frames.append(trial_df)

            sample_df = pd.DataFrame(
                {
                    "block_id": np.repeat(block_id, n_trials * n_samples),
                    "subject_folder": np.repeat(subject_dir.name, n_trials * n_samples),
                    "participant_id": np.repeat(participant_id, n_trials * n_samples),
                    "session_label": np.repeat(session_dir.name, n_trials * n_samples),
                    "session": np.repeat(session_number, n_trials * n_samples),
                    "block": np.repeat(block, n_trials * n_samples),
                    "trial": np.repeat(np.arange(1, n_trials + 1), n_samples),
                    "sample": np.tile(np.arange(1, n_samples + 1), n_trials),
                    "stim_location_deg": stim_in_completed.reshape(-1),
                    "generative_dist": distseqs_completed.reshape(-1),
                    "distribution_switch": pswitch_completed.reshape(-1),
                    "sample_onset_from_trial_s": t_refresh_completed.reshape(-1),
                }
            )
            sample_frames.append(sample_df)

            metadata = {
                "block_id": block_id,
                "subject_folder": subject_dir.name,
                "participant_id": participant_id,
                "session_label": session_dir.name,
                "session": session_number,
                "block": block,
                "n_completed_trials": n_trials,
                "n_planned_trials": n_planned_trials,
                "has_incomplete_behaviour": n_trials != n_planned_trials,
                "n_samples": n_samples,
                "behaviour_file": str(behav_path),
                "sample_seqs_file": str(sample_path),
                "pshort": matlab_value_to_python(sample_mat["pshort"]),
            }
            for struct_name in ["gen", "timing", "stim"]:
                metadata.update(flatten_dict({struct_name: matlab_value_to_python(sample_mat[struct_name])}))
            block_metadata_rows.append(metadata)

    trials_df = pd.concat(trial_frames, ignore_index=True)
    samples_df = pd.concat(sample_frames, ignore_index=True)
    block_metadata_df = pd.DataFrame(block_metadata_rows)
    return trials_df, samples_df, block_metadata_df


In [5]:
def finite_logit(values, eps=1e-12):
    return logit(np.clip(values, eps, 1 - eps))


def P_source_g_samples(samples_arr, mean_neg, mean_pos, std, h, P0, window_size=None):
    P_sample_g_neg = norm(loc=mean_neg, scale=std)
    P_sample_g_pos = norm(loc=mean_pos, scale=std)
    Lneg = P_sample_g_neg.pdf(samples_arr)
    Lpos = P_sample_g_pos.pdf(samples_arr)
    L = np.stack((Lneg, Lpos), axis=1).T
    H = np.array([[1 - h, h], [h, 1 - h]])
    P = P0.copy()
    if window_size is not None:
        L = L[:, -window_size:]
    for ii in range(L.shape[1]):
        P = L[:, ii][:, np.newaxis] * (H @ P)
    return P / P.sum(axis=0, keepdims=True)


def partition_normal(norm_obj, cuts):
    cuts = np.asarray(cuts)
    cdf_values = norm_obj.cdf(cuts)
    return np.diff(np.concatenate(([0], cdf_values, [1])))


def compute_partitioned_Ixy(dist_neg, dist_pos, p_Y, cuts):
    p_XgY0 = partition_normal(dist_neg, cuts)
    p_XgY1 = partition_normal(dist_pos, cuts)
    p_XgY = np.stack((p_XgY0, p_XgY1), axis=1)
    p_XY = p_Y * p_XgY
    p_X = p_XY.sum(axis=1, keepdims=True)
    return np.nansum(p_XY * np.log(p_XY / (p_X * p_Y)))


def map_to_partition(sample_arr, cuts):
    sample_arr = np.asarray(sample_arr)
    bin_edges = np.concatenate(([-np.inf], np.asarray(cuts), [np.inf]))
    return np.digitize(sample_arr, bin_edges) - 1


def P_source_g_partsamples(partsamples_arr, mean_neg, mean_pos, std, cuts, h, P0, window_size=None):
    P_partsamples_g_neg = partition_normal(norm(loc=mean_neg, scale=std), cuts)
    P_partsamples_g_pos = partition_normal(norm(loc=mean_pos, scale=std), cuts)
    Lneg = P_partsamples_g_neg[partsamples_arr]
    Lpos = P_partsamples_g_pos[partsamples_arr]
    L = np.stack((Lneg, Lpos), axis=1).T
    H = np.array([[1 - h, h], [h, 1 - h]])
    P = P0.copy()
    if window_size is not None:
        L = L[:, -window_size:]
    for ii in range(L.shape[1]):
        P = L[:, ii][:, np.newaxis] * (H @ P)
    return P / P.sum(axis=0, keepdims=True)


def encode_partition_sequence(arr):
    if not isinstance(arr, (list, np.ndarray)):
        return np.nan
    arr = np.asarray(arr, dtype=int)
    return int(np.sum(10 ** np.arange(len(arr) - 1, -1, -1) * arr))


def last_k_partition_sequence(arr, k):
    if isinstance(arr, (list, np.ndarray)) and len(arr) >= k:
        return np.asarray(arr)[-k:]
    return np.nan

def P_partsamples_g_source_parallel(partsample_arrs,mean_neg,mean_pos,std,cuts,h,window_size=None):
    P_sample_g_neg = norm(loc=mean_neg, scale=std)
    P_sample_g_pos = norm(loc=mean_pos, scale=std)
    P_partsamples_g_neg = partition_normal(P_sample_g_neg, cuts)
    P_partsamples_g_pos = partition_normal(P_sample_g_pos, cuts)
    Lneg = P_partsamples_g_neg[partsample_arrs]
    Lpos = P_partsamples_g_pos[partsample_arrs]
    L = np.concatenate((Lneg[:,np.newaxis,:],Lpos[:,np.newaxis,:]),axis=1)
    H = np.array([
        [1-h, h],
        [h, 1-h]
    ])
    H = H.T # because we are going "backwards" from source to samples
    P = np.ones((L.shape[0],2,1))
    if window_size is not None:
        L = L[:,:,-window_size:]
    # for ii in range(L.shape[2]-1,0,-1):
    #     P = H @ (L[:,:,ii][:,:,np.newaxis] * P)
    for ii in range(1,L.shape[2]):
        P = H @ (L[:,:,ii-1][:,:,np.newaxis] * P)
    P = L[:,:,L.shape[2]-1][:,:,np.newaxis] * P
    return P.squeeze()

def P_source_g_partsamples_parallel(sample_arrs,mean_neg,mean_pos,std,cuts,h,P0,window_size=None):
    p_XgY = P_partsamples_g_source_parallel(sample_arrs,mean_neg,mean_pos,std,cuts,h,window_size=window_size)
    p_XY = p_XgY * P0.T
    p_YgX = p_XY / p_XY.sum(axis=1,keepdims=True)
    return p_YgX.squeeze()

In [6]:
def identify_hazard_column(block_metadata_df):
    h_col_candidates = [c for c in ["gen.h", "gen.H", "h", "H"] if c in block_metadata_df.columns]
    if h_col_candidates:
        return h_col_candidates[0]
    h_like_cols = [c for c in block_metadata_df.columns if c.lower().endswith(".h") or c.lower() == "h"]
    if len(h_like_cols) != 1:
        raise KeyError(f"Could not uniquely identify hazard-rate column. Candidates: {h_like_cols}")
    return h_like_cols[0]


def compute_optimal_four_partition_cuts():
    dist_neg = norm(loc=mean_neg, scale=std)
    dist_pos = norm(loc=mean_pos, scale=std)

    def obj_function(cut):
        return -compute_partitioned_Ixy(dist_neg, dist_pos, np.array([[0.5, 0.5]]), [-cut, 0, cut])

    result = minimize_scalar(obj_function, bounds=(0, 89), method="bounded")
    cut = result.x
    return [-cut, 0, cut]


def add_partitioned_posterior(trials_df, h_by_block, source_col, suffix, cuts, k):
    seq_col = f"{source_col}_{k}"
    enc_col = f"{source_col}_{k}_enc"
    prob_col = f"post_prob_pos_{suffix}{k}"

    trials_df[seq_col] = trials_df[source_col].apply(lambda arr: last_k_partition_sequence(arr, k))
    trials_df[prob_col] = trials_df.apply(
        lambda row: float(
            P_source_g_partsamples(
                np.asarray(row[seq_col], dtype=int),
                mean_neg,
                mean_pos,
                std,
                cuts,
                float(h_by_block.loc[row["block_id"]]),
                P0,
            )[1, 0]
        )
        if isinstance(row[seq_col], (list, np.ndarray))
        else np.nan,
        axis=1,
    )
    trials_df[enc_col] = trials_df[seq_col].apply(encode_partition_sequence)
    return prob_col, enc_col


def prepare_subject_data(subject_folder):
    trials_df, samples_df, block_metadata_df = load_subject_behavior(subject_folder)

    samples_df.loc[samples_df["stim_location_deg"] == -89.99, "stim_location_deg"] = -90.0
    samples_df.loc[samples_df["stim_location_deg"] == 89.99, "stim_location_deg"] = 90.0

    dist_neg = norm(loc=mean_neg, scale=std)
    dist_pos = norm(loc=mean_pos, scale=std)
    samples_df["LLR_pos"] = np.log(dist_pos.pdf(samples_df["stim_location_deg"]) / dist_neg.pdf(samples_df["stim_location_deg"]))

    group_cols = ["session", "block", "trial"]
    samples_arr_by_trial = (
        samples_df.sort_values("sample")
        .groupby(group_cols)["stim_location_deg"]
        .apply(lambda s: s.dropna().to_numpy())
    )
    trials_df["samples_arr"] = pd.MultiIndex.from_frame(trials_df[group_cols]).map(samples_arr_by_trial)

    llr_arr_by_trial = (
        samples_df.sort_values("sample")
        .groupby(group_cols)["LLR_pos"]
        .apply(lambda s: s.dropna().to_numpy())
    )
    trials_df["LLR_arr"] = pd.MultiIndex.from_frame(trials_df[group_cols]).map(llr_arr_by_trial)

    h_col = identify_hazard_column(block_metadata_df)
    h_by_block = block_metadata_df.set_index("block_id")[h_col]

    trials_df["post_prob_pos"] = trials_df.apply(
        lambda row: float(
            P_source_g_samples(
                np.asarray(row["samples_arr"], dtype=float),
                mean_neg,
                mean_pos,
                std,
                float(h_by_block.loc[row["block_id"]]),
                P0,
            )[1, 0]
        ),
        axis=1,
    )
    trials_df["post_prob_1b_pos"] = trials_df.apply(
        lambda row: float(
            P_source_g_samples(
                np.asarray(row["samples_arr"], dtype=float),
                mean_neg,
                mean_pos,
                std,
                float(h_by_block.loc[row["block_id"]]),
                P0,
                window_size=1,
            )[1, 0]
        ),
        axis=1,
    )
    trials_df["LPO_pos"] = finite_logit(trials_df["post_prob_pos"].to_numpy())
    trials_df["LPO_1b_pos"] = finite_logit(trials_df["post_prob_1b_pos"].to_numpy())

    cuts = compute_optimal_four_partition_cuts()
    trials_df["partsamples_arr"] = trials_df["samples_arr"].apply(lambda arr: map_to_partition(np.asarray(arr), cuts))
    add_partitioned_posterior(trials_df, h_by_block, "partsamples_arr", "partsample", cuts, 3)
    add_partitioned_posterior(trials_df, h_by_block, "partsamples_arr", "partsample", cuts, 5)

    cuts3 = [0]
    trials_df["part2samples_arr"] = trials_df["samples_arr"].apply(lambda arr: map_to_partition(np.asarray(arr), cuts3))
    add_partitioned_posterior(trials_df, h_by_block, "part2samples_arr", "part2sample", cuts3, 5)
    add_partitioned_posterior(trials_df, h_by_block, "part2samples_arr", "part2sample", cuts3, 7)

    return trials_df, samples_df, block_metadata_df


In [9]:
from itertools import product

Xset = np.array(list(product(range(4), repeat=5)))
Xset.shape

(1024, 5)

In [23]:
cuts = compute_optimal_four_partition_cuts()
p_YgX = P_source_g_partsamples_parallel(Xset, mean_neg, mean_pos, std, cuts, h=h, P0=P0)
p_YgX[:16]

array([[0.98144445, 0.01855555],
       [0.93970175, 0.06029825],
       [0.85007415, 0.14992585],
       [0.62555298, 0.37444702],
       [0.97359279, 0.02640721],
       [0.91570403, 0.08429597],
       [0.79807115, 0.20192885],
       [0.53799794, 0.46200206],
       [0.95491956, 0.04508044],
       [0.86190242, 0.13809758],
       [0.69425859, 0.30574141],
       [0.4008578 , 0.5991422 ],
       [0.89338803, 0.10661197],
       [0.71173556, 0.28826444],
       [0.47321353, 0.52678647],
       [0.20928418, 0.79071582]])

In [ ]:
import utilities as utils

p_XgY = P_partsamples_g_source_parallel(Xset, mean_neg, mean_pos, std, cuts, h=h, window_size=None)
p_Y = np.array([[0.5, 0.5]])
IBbound = utils.get_IB_bound(p_XgY, p_Y, N_threads=14)

In [ ]:
plt.plot(IBbound['I_XR'], IBbound['I_YR'], label='IB Bound')

In [ ]:
participant_id = "DCB"
trials_df, samples_df, block_metadata_df = prepare_subject_data(participant_id)

Xemp = np.stack(trials_df["partsamples_arr"].to_numpy())
Yemp = np.stack(trials_df["correct_response_dist"].to_numpy())
IBbound_emp = utils.get_IB_emp(Xemp, Xset, Yemp, p_XgY, p_Y,N_threads=14)
